# Tech Challenge — Fase 2 | Desenvolvimento do Modelo

**Projeto:** Classificação binária de qualidade do vinho — FIAP Pós-Tech Data Analytics.

**Dataset Original:** <https://www.kaggle.com/datasets/yasserh/wine-quality-dataset>

**Objetivo:** Treinar de classificação, e comparar o desempenho
entre eles. Avaliar os modelos utilizando métricas adequadas.
Comparar os resultados entre os modelos testados.

# 1. Preparação do Ambiente de Treinamento

Antes de começar com o treino e avaliação dos modelos, iremos preparar os dados importando nosso dataframe original e o dividido em treino e teste, no notebook 02.

In [5]:
import pandas as pd
df_original = pd.read_csv('/content/WineQTv2.csv')
df_divisao = pd.read_csv('/content/divisao_treino_teste.csv')

In [6]:
df_original.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [7]:
df_divisao.head()

,indice_original,Id,quality_class,conjunto
0,0,0,0,treino
1,1,1,0,treino
2,2,2,0,treino
3,3,3,0,treino
4,4,4,0,treino


In [8]:
# Retornando o índice original
df_original['indice_original'] = df_original.index

In [9]:
# Junção do df original com o df de treino e teste
df_completo = pd.merge(df_original, df_divisao, on='indice_original', how='inner')

In [10]:
df_completo.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,indice_original,Id,quality_class,conjunto
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,0,0,treino
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1,1,0,treino
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2,2,0,treino
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3,3,0,treino
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4,4,0,treino


In [11]:
# Limpando colunas redundantes
df_completo = df_completo.drop(columns=['quality', 'Id', 'indice_original'])

In [12]:
# Filtragem dos dados completos em treino e teste
df_treino = df_completo[df_completo['conjunto'] == 'treino']
df_teste = df_completo[df_completo['conjunto'] == 'teste']

In [13]:
# Definição das features (X) e a variável alvo (y)
X_train = df_treino.drop(columns=['conjunto', 'quality_class'])
y_train = df_treino['quality_class']

X_test = df_teste.drop(columns=['conjunto', 'quality_class'])
y_test = df_teste['quality_class']

# 2. Escolhendo os Algoritmos

Iremos testar 3 algoritmos para avaliar os resultados e escolher o mais satisfatório: Regressão Logística, Random Forest e KNN.

## 2.1 Regressão Logística

In [14]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

In [15]:
# Isolar grupos com a mesma química para evitar vazamento
grupos = X_train.groupby(X_train.columns.tolist()).ngroup()

In [16]:
# Criação da pipeline de regressão logistica (lr)
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

In [17]:
# Criação da estratégia de Validação Cruzada (cv)
estrategia_cv = StratifiedGroupKFold(n_splits=5)

In [18]:
metricas = ['f1', 'precision', 'recall', 'roc_auc']

In [19]:
resultados_cv = cross_validate(
    pipeline_lr,
    X_train,
    y_train,
    groups=grupos,
    cv=estrategia_cv,
    scoring=metricas
)

In [20]:
print("=== Resultados da Regressão Logística (Baseline) ===")
print(f"F1-Score Médio:  {np.mean(resultados_cv['test_f1']):.4f}")
print(f"Precision Média: {np.mean(resultados_cv['test_precision']):.4f}")
print(f"Recall Médio:    {np.mean(resultados_cv['test_recall']):.4f}")
print(f"ROC-AUC Média:   {np.mean(resultados_cv['test_roc_auc']):.4f}")

=== Resultados da Regressão Logística (Baseline) ===
F1-Score Médio:  0.4958
Precision Média: 0.3655
Recall Médio:    0.7785
ROC-AUC Média:   0.8622


Avaliação dos Resultados:

Recall (0,77): O modelo detectou corretamente 77% de todos os vinhos que realmente são de alta qualidade.

Precision (0,36): É o ponto fraco da regressão. Quando o algoritmo classifica um vinho como sendo de "alta qualidade", ele acerta apenas em 36% das vezes. O restante compõe um alto volume de falsos positivos (vinhos comuns classificados erroneamente como premium).

F1-Score (0,49): Sendo a média harmônica entre Recall e Precision, este valor mediano reflete a penalização matemática sofrida pela baixa precisão.

ROC-AUC (0,86): O valor indica que, se o modelo precisar ordenar um vinho premium e um comum, ele tem 86% de chance de atribuir uma probabilidade maior ao vinho premium.

## 2.2 Random Forest

In [21]:
from sklearn.ensemble import RandomForestClassifier

In [22]:
# Criação do pipeline de random forest (rf)
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced_subsample',
        random_state=42
    ))
])

In [23]:
resultados_rf = cross_validate(
    pipeline_rf,
    X_train,
    y_train,
    groups=grupos,
    cv=estrategia_cv,
    scoring=metricas
)

In [24]:
print("=== Resultados do Random Forest ===")
print(f"F1-Score Médio:  {np.mean(resultados_rf['test_f1']):.4f}")
print(f"Precision Média: {np.mean(resultados_rf['test_precision']):.4f}")
print(f"Recall Médio:    {np.mean(resultados_rf['test_recall']):.4f}")
print(f"ROC-AUC Média:   {np.mean(resultados_rf['test_roc_auc']):.4f}")

=== Resultados do Random Forest ===
F1-Score Médio:  0.4409
Precision Média: 0.7637
Recall Médio:    0.3142
ROC-AUC Média:   0.8680


Avaliação dos Resultados:

Precision (76%): Alta confiabilidade. Quando o modelo afirma que um vinho é premium, acerta na grande maioria das vezes. Evita falsos positivos e não desperdiça o tempo do especialista humano.

Recall (31%): Baixa captura, o algoritmo deixa passar quase 70% das garrafas de alta qualidade do estoque (falsos negativos).

F1-Score (44%): O valor baixo evidencia que o modelo falha na visão de negócio, pois sacrifica demais o rastreio (Recall) apenas para manter a precisão alta.

ROC-AUC (86%): O valor alto confirma que o modelo aprendeu perfeitamente a diferença química entre os vinhos comuns e premium; o baixo desempenho nas outras métricas pode ser culpa da régua do corte limiar padrão (50%).

## 2.3 KNN (K-Nearest Neighbors)

In [25]:
from sklearn.neighbors import KNeighborsClassifier

In [26]:
# Criação do pipeline de knn
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(n_neighbors=5, weights='distance'))
])

In [27]:
resultados_knn = cross_validate(
    pipeline_knn,
    X_train,
    y_train,
    groups=grupos,
    cv=estrategia_cv,
    scoring=metricas
)

In [28]:
print("=== Resultados do KNN ===")
print(f"F1-Score Médio:  {np.mean(resultados_knn['test_f1']):.4f}")
print(f"Precision Média: {np.mean(resultados_knn['test_precision']):.4f}")
print(f"Recall Médio:    {np.mean(resultados_knn['test_recall']):.4f}")
print(f"ROC-AUC Média:   {np.mean(resultados_knn['test_roc_auc']):.4f}")

=== Resultados do KNN ===
F1-Score Médio:  0.4341
Precision Média: 0.5002
Recall Médio:    0.3914
ROC-AUC Média:   0.7582


Avaliação dos Resultados:

Precision (50%): Metade dos vinhos classificados como premium pelo modelo eram (falsos positivos).

Recall (39%): Captura ineficiente. O algoritmo perde o rastro de aproximadamente 60% das garrafas de alta qualidade do estoque (falsos negativos).

F1-Score (43%): Nota de equilíbrio baixa. Confirma que o modelo falha tanto na precisão do alerta quanto na abrangência da busca.

ROC-AUC (75%): Dificuldade de distinção espacial. Indica que medir a similaridade química apenas pela distância geométrica entre os vinhos não foi suficiente para criar uma fronteira clara que isole a classe premium.

## 2.4 Algoritmo Escolhido

O Algoritmo escolhido foi o Random Forest por apresentar a maior taxa de precisão (77%). O algoritmo demonstra uma excelente capacidade de distinção entre as classes, alcançando um forte ROC-AUC de 86%. As outras métricas, como Recall de 31% e F1-Score de 42% ainda podem ser melhoradas através de um ajustes no algoritmo.

A Regressão Logística apesar de ter uma boa captação (Recall 77%) falhou ao gerar um volume excessivo de falsos positivos, o que derrubou sua Precision para apenas 36% e resultou em um F1-Score mediano de 49%.

O KNN, por depender da distância geométrica entre os dados, sofreu severamente com o desbalanceamento da base. No geral teve o pior desempenho.

# 3. Configuração do Grid de Hiperparâmetros

Nesta etapa iremos variar o número de árvores, limitar a profundidade máxima (para evitar o sobreajuste nos dados da classe majoritária) e alterar a quantidade mínima de amostras exigidas para ramificar uma decisão.

In [29]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [30]:
# Pipeline de random forest
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(class_weight='balanced_subsample', random_state=42))
])

In [31]:
# Definindo as combinações que o Scikit-Learn irá testar
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 5, 10, 15],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

In [32]:
# Utilizando o F1-Score como a métrica principal para escolher o modelo vencedor
grid_search = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid,
    cv=estrategia_cv,
    scoring='f1',
    n_jobs=-1
)

In [33]:
# Rodando o treinamento em todos os grupos para respeitar a regra de não-vazamento
grid_search.fit(X_train, y_train, groups=grupos)

GridSearchCV(cv=StratifiedGroupKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier',
                                        RandomForestClassifier(class_weight='balanced_subsample',
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [None, 5, 10, 15],
                         'classifier__min_samples_leaf': [1, 2, 4],
                         'classifier__min_samples_split': [2, 5, 10],
                         'classifier__n_estimators': [100, 200, 300]},
             scoring='f1')

In [34]:
print("=== Melhor Configuração Encontrada ===")
print(grid_search.best_params_)
print(f"Melhor F1-Score na Validação Cruzada: {grid_search.best_score_:.4f}")

=== Melhor Configuração Encontrada ===
{'classifier__max_depth': 10, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 100}
Melhor F1-Score na Validação Cruzada: 0.5511


Para combater o risco de sobreajuste (overfitting) natural dos algoritmos baseados em árvores em bases desbalanceadas, foi aplicada uma busca exaustiva via GridSearchCV focada em maximizar o F1-Score.

A configuração vencedora (max_depth=10, min_samples_leaf=4, min_samples_split=10, n_estimators=100) estabilizou o aprendizado ao limitar a profundidade das ramificações e ao exigir um volume mínimo de amostras para cada decisão final.

Essa arquitetura garantiu que o Random Forest generalizasse os padrões químicos em vez de apenas decorá-los, consolidando um F1-Score interno de 0,5511 durante o ciclo rigoroso de validação cruzada.

## Avaliação: Performance do Modelo com os Hiperparâmetros

Nesta etapa vamos fazer a avaliação do algoritmo com os hiperparâmetros definidos anteriormente. Esse passo mede a capacidade real de generalização do algoritmo, provando se ele realmente aprendeu as regras químicas ou se apenas decorou os dados de treinamento.


In [35]:
from sklearn.metrics import classification_report, confusion_matrix

In [36]:
# Rastreando o melhor modelo em todo o X_train
melhor_modelo = grid_search.best_estimator_

In [37]:
# Fazendo as predições finais na base de teste isolada
y_pred_final = melhor_modelo.predict(X_test)

In [38]:
print("\n=== Matriz de Confusão no Teste ===")
print(confusion_matrix(y_test, y_pred_final))


=== Matriz de Confusão no Teste ===
[[179  18]
 [ 12  20]]


In [39]:
print("\n=== Relatório de Classificação no Teste ===")
print(classification_report(y_test, y_pred_final))


=== Relatório de Classificação no Teste ===
              precision    recall  f1-score   support

           0       0.94      0.91      0.92       197
           1       0.53      0.62      0.57        32

    accuracy                           0.87       229
   macro avg       0.73      0.77      0.75       229
weighted avg       0.88      0.87      0.87       229



Aplicado sobre dados novos, o modelo otimizado demonstrou um ganho em relação à sua versão padrão. A taxa de captura (Recall) da classe minoritária saltou para 62%, garantindo a identificação da maior parte do estoque de alta qualidade.

O impacto esperado dessa flexibilização foi o recuo da Precision para 53%, gerando um volume controlável de falsos positivos.

O F1-Score definitivo de 0,57 no teste isolado confirma que a utilização do GridSearchCV ajudou a evitar o sobreajuste, entregando um classificador mais equilibrado.

# Ajuste do Limiar de Decisão





Vamos ajustar o limiar do algoritmo, para aumentar a sensibilidade do modelo e equilibrar a quantidade de falsos negativos e falsos positivo, buscando uma confiabilidade maior do modelo.

In [40]:
#  Calcular qual é o melhor valor de limiar
from sklearn.metrics import precision_recall_curve

In [41]:
# Extrair as probabilidades brutas para a classe 1 (alta qualidade)
probabilidades = melhor_modelo.predict_proba(X_test)[:, 1]

In [42]:
# Calcular a precisão e o recall para todos os limiares possíveis
precisions, recalls, thresholds = precision_recall_curve(y_test, probabilidades)

In [43]:
# Calcular o F1-Score para cada limiar da curva
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

In [44]:
# Encontrar o índice onde o F1-Score atinge o seu pico máximo
melhor_indice = np.argmax(f1_scores)
melhor_limiar = thresholds[melhor_indice]
melhor_f1 = f1_scores[melhor_indice]

print(f"=== Melhores Valores Alcançados ===")
print(f"Melhor Limiar de Decisão: {melhor_limiar * 100:.1f}%")
print(f"F1-Score Máximo Alcançado: {melhor_f1:.4f}")
print(f"Precision neste ponto: {precisions[melhor_indice]:.4f}")
print(f"Recall neste ponto: {recalls[melhor_indice]:.4f}")

=== Melhores Valores Alcançados ===
Melhor Limiar de Decisão: 43.2%
F1-Score Máximo Alcançado: 0.6829
Precision neste ponto: 0.5600
Recall neste ponto: 0.8750


## Avaliação: Limiar de Decisão

Como vimos anteriormente, o valor de limiar de decisão é de aproximadamente 43%. Então agora iremos fazer a avaliação do modelo com o novo limiar.

In [45]:
probabilidades = melhor_modelo.predict_proba(X_test)[:, 1]

limiar = 0.43

y_pred_ajustado = (probabilidades >= limiar).astype(int)

In [46]:
print(f"=== Matriz de Confusão (Limiar {limiar*100}%) ===")
print(confusion_matrix(y_test, y_pred_ajustado))

print(f"\n=== Relatório de Classificação (Limiar {limiar*100}%) ===")
print(classification_report(y_test, y_pred_ajustado))

=== Matriz de Confusão (Limiar 43.0%) ===
[[174  23]
 [  4  28]]

=== Relatório de Classificação (Limiar 43.0%) ===
              precision    recall  f1-score   support

           0       0.98      0.88      0.93       197
           1       0.55      0.88      0.67        32

    accuracy                           0.88       229
   macro avg       0.76      0.88      0.80       229
weighted avg       0.92      0.88      0.89       229



A aplicação da Curva Precision-Recall ajudou a revelar o melhor ponto de equilíbrio exato do algoritmo. Enquanto o corte padrão de 50% era rígido demais, fazendo perder garrafas premiums, o novo limiar em 43% entregou a melhor eficiência operacional para o negócio:

Recall de 88%: O modelo localizou com sucesso 28 das 32 garrafas de alta qualidade. Deixar passar apenas 4 garrafas premium (falsos negativos) representa uma perda de oportunidade baixa frente ao volume total do estoque.

Precision de 55%: A configuração conteve a geração de alarmes falsos para apenas 23 garrafas comuns (falsos positivos).

F1-Score de 0,67: A evolução da média harmônica de 0,57 (no modelo padrão) para o pico de 0,67 comprova que a técnica melhorou a performance das variáveis disponíveis.

# Inferência

Com o modelo finalizado, iremos fazer um teste de inferência para testar o comportamento de novos dados.

In [47]:
# Colocar dados novos para verificar a saída e a confiabilidade
dados_novo_vinho = pd.DataFrame([{
    'fixed acidity': 8.5,
    'volatile acidity': 0.28,
    'citric acid': 0.56,
    'residual sugar': 1.8,
    'chlorides': 0.092,
    'free sulfur dioxide': 35.0,
    'total sulfur dioxide': 103.0,
    'density': 0.9969,
    'pH': 3.30,
    'sulphates': 0.75,
    'alcohol': 10.5
}])

In [48]:
# Utilizando o StandardScaler para padronização automática dos novos dados
probabilidade_premium = melhor_modelo.predict_proba(dados_novo_vinho)[0, 1]

limiar_negocio = 0.43

In [49]:
print("=== Sistema de Classificação de Qualidade ===")
if probabilidade_premium >= limiar_negocio:
    print(f"Aleta: Vinho de ALTA QUALIDADE detectado.")
    print(f"Confiança do algoritmo: {probabilidade_premium * 100:.1f}%")
else:
    print(f"Status: Vinho de qualidade COMUM.")
    print(f"Confiança do algoritmo: {probabilidade_premium * 100:.1f}%")

=== Sistema de Classificação de Qualidade ===
Aleta: Vinho de ALTA QUALIDADE detectado.
Confiança do algoritmo: 61.0%


# Conclusão

Com o modelo finalizado, podemos ver que foi possível automatizar a triagem dos melhores vinhos, mesmo lidando com um dataset original desbalanceado. Nosso banco de dados tinha apenas ≈14% das garrafas sendo realmente premium, o que naturalmente interferiu na capacidade de aprendizado inicial do algoritmo. Ainda assim, o Random Forest foi o algoritmo escolhido porque conseguiu mapear as misturas complexas da química do vinho, mostrando um maior desempenho em relação a Regressão Logística e o KNN.

Para melhorar ainda mais nosso modelo, fizemos um refinamento calculando e aplicando o melhor limiar de decisão para o nosso caso. Ajustando a régua de decisão para 43%, encontramos o melhor ponto de equilíbrio. O modelo consegue agora abranger 88% de toda a reserva de melhores vinhos sem disparar falsos alarmes o tempo todo, sendo bem mais eficiente.

Mesmo com esse salto na confiabilidade do modelo, encontramos uma limitação de em relação a falta de indicadores sobre a qualidade percebida ao paladar humano, já que o algoritmo avalia apenas a composição química. Para uma versão 2.0, a empresa poderia coletar detalhes sobre a origem como o tipo de uva, o clima da safra e o tempo em barril, ou quaisquer coisas que possam afetar o sabor do vinho e impactar na sua nota final, deixando o modelo ainda mais inteligente.